In [55]:
# ==============================================================================
# NOTEBOOK SETUP - Interaktive Analyse
# ==============================================================================
from pathlib import Path

# Definiere Datenpfade hier (absolute Pfade für stabile Notebook-Ausführung)
raw_data_dir = "/work/pyhton-analysis/BA/detector evaluation/raw_data_overhead_evaluation_20260303_212221"
output_csv = "/work/pyhton-analysis/BA/overhead_results_20260303_notebook.csv"

print(f"Raw data directory: {raw_data_dir}")
print(f"Output CSV:         {output_csv}")
print()

Raw data directory: /work/pyhton-analysis/BA/detector evaluation/raw_data_overhead_evaluation_20260303_212221
Output CSV:         /work/pyhton-analysis/BA/overhead_results_20260303_notebook.csv



In [61]:
import statistics


def werte_fuer_metric(results, mode, key):
    werte = []
    for eintrag in results:
        if eintrag.get("mode") == mode and eintrag.get(key) is not None:
            werte.append(float(eintrag[key]))
    return werte


def relative_aenderung(neu, alt):
    if alt == 0:
        return float("nan")
    return ((neu - alt) / alt) * 100.0


def berechne_cpu_overhead(detector_cpu, baseline_cpu):
    """
    Positiv = mehr CPU-Verbrauch durch den Detektor
    """
    return relative_aenderung(detector_cpu, baseline_cpu)


def berechne_io_overhead_iops(baseline_iops, detector_iops):
    """
    Positiv = IOPS-Verlust durch den Detektor
    """
    if baseline_iops == 0:
        return float("nan")
    return ((baseline_iops - detector_iops) / baseline_iops) * 100.0


def berechne_tabellenwerte_durchschnitt(results):
    # Laufzeit
    baseline_runtime = werte_fuer_metric(results, "baseline", "fio_runtime_sec")
    detector_runtime = werte_fuer_metric(results, "detector", "fio_runtime_sec")

    # CPU
    baseline_cpu = werte_fuer_metric(results, "baseline", "nfsd_avg_cpu_pct")
    detector_cpu = werte_fuer_metric(results, "detector", "nfsd_avg_cpu_pct")

    # Latenz
    baseline_lat_ns = werte_fuer_metric(results, "baseline", "fio_lat_ns")
    detector_lat_ns = werte_fuer_metric(results, "detector", "fio_lat_ns")

    # IOPS
    baseline_iops = werte_fuer_metric(results, "baseline", "fio_iops")
    detector_iops = werte_fuer_metric(results, "detector", "fio_iops")

    if not baseline_runtime or not detector_runtime:
        raise ValueError("Es fehlen Laufzeitwerte.")
    if not baseline_cpu or not detector_cpu:
        raise ValueError("Es fehlen CPU-Werte.")
    if not baseline_lat_ns or not detector_lat_ns:
        raise ValueError("Es fehlen Latenzwerte.")
    if not baseline_iops or not detector_iops:
        raise ValueError("Es fehlen IOPS-Werte.")

    # Durchschnittswerte
    runtime_baseline = statistics.mean(baseline_runtime)
    runtime_detector = statistics.mean(detector_runtime)

    cpu_baseline = statistics.mean(baseline_cpu)
    cpu_detector = statistics.mean(detector_cpu)

    iops_baseline = statistics.mean(baseline_iops)
    iops_detector = statistics.mean(detector_iops)

    # ns -> µs
    baseline_lat_us = [wert / 1000.0 for wert in baseline_lat_ns]
    detector_lat_us = [wert / 1000.0 for wert in detector_lat_ns]

    latenz_baseline = statistics.mean(baseline_lat_us)
    latenz_detector = statistics.mean(detector_lat_us)

    # Overheads
    cpu_overhead = berechne_cpu_overhead(cpu_detector, cpu_baseline)
    io_overhead = berechne_io_overhead_iops(iops_baseline, iops_detector)

    return {
        "laufzeit": {
            "baseline": runtime_baseline,
            "detector": runtime_detector,
            "absolute_aenderung": runtime_detector - runtime_baseline,
            "relative_aenderung": relative_aenderung(runtime_detector, runtime_baseline),
            "einheit": "s"
        },
        "cpu": {
            "baseline": cpu_baseline,
            "detector": cpu_detector,
            "absolute_aenderung": cpu_detector - cpu_baseline,
            "relative_aenderung": relative_aenderung(cpu_detector, cpu_baseline),
            "overhead": cpu_overhead,
            "einheit": "%"
        },
        "latenz": {
            "baseline": latenz_baseline,
            "detector": latenz_detector,
            "absolute_aenderung": latenz_detector - latenz_baseline,
            "relative_aenderung": relative_aenderung(latenz_detector, latenz_baseline),
            "einheit": "µs"
        },
        "iops": {
            "baseline": iops_baseline,
            "detector": iops_detector,
            "absolute_aenderung": iops_detector - iops_baseline,
            "relative_aenderung": relative_aenderung(iops_detector, iops_baseline),
            "overhead": io_overhead,
            "einheit": "IOPS"
        }
    }


def drucke_tabellenwerte(tabellenwerte):
    print("Mittelwert Laufzeit [s]")
    print(f"  Baseline:            {tabellenwerte['laufzeit']['baseline']:.3f}")
    print(f"  Detector:            {tabellenwerte['laufzeit']['detector']:.3f}")
    print(f"  Absolute Änderung:   {tabellenwerte['laufzeit']['absolute_aenderung']:+.3f} s")
    print(f"  Relative Änderung:   {tabellenwerte['laufzeit']['relative_aenderung']:+.2f} %")

    print("\nMittelwert nfsd CPU [%]")
    print(f"  Baseline:            {tabellenwerte['cpu']['baseline']:.3f}")
    print(f"  Detector:            {tabellenwerte['cpu']['detector']:.3f}")
    print(f"  Absolute Änderung:   {tabellenwerte['cpu']['absolute_aenderung']:+.3f} Prozentpunkte")
    print(f"  Relative Änderung:   {tabellenwerte['cpu']['relative_aenderung']:+.2f} %")
    print(f"  CPU-Overhead:        {tabellenwerte['cpu']['overhead']:+.2f} %")

    print("\nMittelwert Latenz [µs]")
    print(f"  Baseline:            {tabellenwerte['latenz']['baseline']:.3f}")
    print(f"  Detector:            {tabellenwerte['latenz']['detector']:.3f}")
    print(f"  Absolute Änderung:   {tabellenwerte['latenz']['absolute_aenderung']:+.3f} µs")
    print(f"  Relative Änderung:   {tabellenwerte['latenz']['relative_aenderung']:+.2f} %")

    print("\nMittelwert IOPS")
    print(f"  Baseline:            {tabellenwerte['iops']['baseline']:.3f}")
    print(f"  Detector:            {tabellenwerte['iops']['detector']:.3f}")
    print(f"  Absolute Änderung:   {tabellenwerte['iops']['absolute_aenderung']:+.3f} IOPS")
    print(f"  Relative Änderung:   {tabellenwerte['iops']['relative_aenderung']:+.2f} %")
    print(f"  I/O-Overhead:        {tabellenwerte['iops']['overhead']:+.2f} %")


def drucke_kurzfazit(tabellenwerte):
    print("\n" + "=" * 72)
    print("KURZFAZIT")
    print("=" * 72)
    print(
        f"Der durchschnittliche CPU-Overhead beträgt "
        f"{tabellenwerte['cpu']['overhead']:+.2f} %."
    )
    print(
        f"Der durchschnittliche I/O-Overhead auf Basis der IOPS beträgt "
        f"{tabellenwerte['iops']['overhead']:+.2f} %."
    )


# Beispiel:
# tabellenwerte = berechne_tabellenwerte_durchschnitt(results)
# drucke_tabellenwerte(tabellenwerte)
# drucke_kurzfazit(tabellenwerte)

In [62]:
tabellenwerte = berechne_tabellenwerte_durchschnitt(results)
drucke_tabellenwerte(tabellenwerte)

Mittelwert Laufzeit [s]
  Baseline:            320.343
  Detector:            320.642
  Absolute Änderung:   +0.300 s
  Relative Änderung:   +0.09 %

Mittelwert nfsd CPU [%]
  Baseline:            7.367
  Detector:            7.413
  Absolute Änderung:   +0.046 Prozentpunkte
  Relative Änderung:   +0.63 %
  CPU-Overhead:        +0.63 %

Mittelwert Latenz [µs]
  Baseline:            715.456
  Detector:            715.876
  Absolute Änderung:   +0.420 µs
  Relative Änderung:   +0.06 %

Mittelwert IOPS
  Baseline:            805.869
  Detector:            805.113
  Absolute Änderung:   -0.757 IOPS
  Relative Änderung:   -0.09 %
  I/O-Overhead:        +0.09 %
